# Liver: selected supplementary controls

Replay the saved nested selections, five-seed CNN stability, graph
ablations and two complete mouse-isolated representation controls.
The [main notebook](mantpy_liver_full.ipynb) remains the primary analysis.
No parameter search runs here.

**Runtime:** several hours to more than a day is possible. The selected
nested models include 200–500 epochs, widths up to 256, radius graphs and
GAT/GIN. Eight additional training-mouse-only CNNs each use 200 epochs.
This complete control notebook is substantially longer than the main
notebook and may require multiple ordinary Colab sessions. Completed
fold/seed and encoder checkpoints are reused. Times are recorded by stage.
Select T4 GPU in Colab for the CNNs.

In [ ]:
from pathlib import Path
import hashlib
from urllib.request import urlopen

_SETUP = Path("experiment_setup.py")
_SETUP_SHA256 = "6faab2e74f2559b2cc76cbb5b0323215a58e15ab2fb355050c55822de4214175"
if not _SETUP.exists():
    with urlopen("https://raw.githubusercontent.com/moeghaf/mantpy_reproducibility/mantpy-api-v1.1.0/notebooks/experiments/experiment_setup.py", timeout=90) as response:
        _setup_bytes = response.read()
    if hashlib.sha256(_setup_bytes).hexdigest() != _SETUP_SHA256:
        raise RuntimeError("Downloaded setup checksum does not match this notebook.")
    _setup_temp = _SETUP.with_name(_SETUP.name + ".download")
    _setup_temp.write_bytes(_setup_bytes)
    _setup_temp.replace(_SETUP)
if hashlib.sha256(_SETUP.read_bytes()).hexdigest() != _SETUP_SHA256:
    raise RuntimeError("Use the setup file matching this notebook release.")
from experiment_setup import prepare_experiment
ASSETS = prepare_experiment("liver")


In [ ]:
import os
from pathlib import Path
import time

import mantpy as mt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from liver_analysis import data, descriptive
import liver_helpers as lh

torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SELECTED = lh.read_selected()
SEEDS = SELECTED["seeds"]                 # five training seeds: 0–4
RESULTS = Path("results/liver")
RESULTS.mkdir(parents=True, exist_ok=True)
TIMINGS = {}
started = time.perf_counter()
print(f"CNN: {DEVICE}; GNN: CPU; {len(SEEDS)} training seeds")

In [ ]:
CONTROL_RESULTS = Path("results/liver_controls")
CONTROL_RESULTS.mkdir(parents=True, exist_ok=True)
PARTITION_MODE = "fresh"                 # match the main notebook, or explicitly "paper"
MICE = sorted(SELECTED["nested_gnn"])
CONTROL_METRICS = []

In [ ]:
stage = time.perf_counter()
liver = mt.datasets.schistosoma_ecm(
    cache_dir=os.environ.get("MANTPY_CACHE"), progressbar=True, verbose=False
)
images, meta = data.cohort(liver)
assert len(images) == 24 and meta.mouse.nunique() == 8
TIMINGS["load_data"] = time.perf_counter() - stage
display(data.mouse_table(meta))

In [ ]:
import json

partition_dir = RESULTS / PARTITION_MODE
if PARTITION_MODE == "paper" and not (partition_dir / "partition.json").exists():
    # The deposited labels need raw coordinates and marker summaries, not a new CNN fit.
    stage = time.perf_counter()
    norm, bounds = data.normalise(images)
    nodes = data.patch_features(norm, meta)
    nodes["signal"] = data.signal_mask(nodes)
    nodes = lh.use_partition(nodes, None, mode="paper")
    TIMINGS["paper_partition_preparation"] = time.perf_counter() - stage
else:
    record = json.loads((partition_dir / "partition.json").read_text())
    for name, field in [("nodes.csv.gz", "nodes_sha256"), ("metadata.csv", "metadata_sha256")]:
        if lh.file_hash(partition_dir / name) != record[field]:
            raise ValueError(f"Changed main-experiment input: {name}")
    nodes = pd.read_csv(partition_dir / "nodes.csv.gz")
    meta = pd.read_csv(partition_dir / "metadata.csv")
print(f"Partition: {PARTITION_MODE}; {int(nodes.signal.sum()):,} signal patches")

## 1. Replay the eight nested-selected configurations

Each selection was chosen inside its corresponding outer training fold.
Refit that configuration with five seeds on the seven training mice and
evaluate its held-out mouse. Preserve its radius graph, architecture,
regularization and original patch-pooled loss. This comparison is separate
from the fixed, ROI-mean primary prediction.

In [ ]:
selection_table = pd.DataFrame([
    {"held_mouse": mouse, "selected_trial": record["trial"],
     "inner_objective": record["inner_objective"], **record["config"]}
    for mouse, record in SELECTED["nested_gnn"].items()
])
display(selection_table)
selection_table.to_csv(CONTROL_RESULTS / "nested_selected_settings.csv", index=False)

In [ ]:
# Recorded outer folds use the public single-fit trainer; original selections stay fixed.
stage = time.perf_counter()
nested_folds = []
for mouse in MICE:
    config = SELECTED["nested_gnn"][mouse]["config"]
    _, selected_graphs = lh.make_graphs(nodes, meta, graph_type=config["graph_type"])
    result = lh.run_outer_folds(
        selected_graphs, meta, held_mice=[mouse], seeds=SEEDS, config=config,
        objective="patch_sum", output_dir=CONTROL_RESULTS / PARTITION_MODE / "nested"
    )
    nested_folds.append(result)
    lh.save_outer_outputs(result, CONTROL_RESULTS / PARTITION_MODE / "nested" / mouse)
    del selected_graphs
nested = lh.merge_outer_results(nested_folds)
lh.save_outer_outputs(nested, CONTROL_RESULTS / PARTITION_MODE / "nested")
CONTROL_METRICS.append(lh.metric_table(nested["ensemble"], "Nested selections · patch-pooled"))
TIMINGS["nested_40_fits"] = time.perf_counter() - stage
display(CONTROL_METRICS[-1])

## 2. Selected encoder stability

Fit seeds 0–4 at the same selected settings. Reuse the completed seed-0
encoder from the main notebook, fit four more encoders, and cluster each
embedding separately at K=6. Compare the partitions on the common
recomputed cohort signal set. This statistic uses fresh partitions even
when classifier replay uses the deposited partition.

In [ ]:
stage = time.perf_counter()
norm, bounds = data.normalise(images)
fresh_nodes = data.patch_features(norm, meta)
fresh_nodes["signal"] = data.signal_mask(fresh_nodes)
windows = lh.make_windows(norm, fresh_nodes)
labels_by_seed = {}
for seed in SEEDS:
    cnn, raw, _, record = lh.fit_cnn_cached(
        windows, seed=seed, config=SELECTED["cnn"],
        output_dir=RESULTS, device=DEVICE
    )
    _, labels_by_seed[seed] = lh.cluster_embedding(raw, seed=seed, K=6)
    np.save(CONTROL_RESULTS / f"fresh_labels_seed{seed}.npy", labels_by_seed[seed])
    del cnn, raw
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
stability = lh.stability_table(labels_by_seed)
stability.to_csv(CONTROL_RESULTS / "selected_encoder_stability.csv", index=False)
display(stability)
print(f"Fresh mean pairwise ARI: {stability.ARI.mean():.3f}")
print(f"Deposited paper reference: {SELECTED['paper_reference']['selected_encoder_five_seed_stability']}")
TIMINGS["five_seed_encoder_stability"] = time.perf_counter() - stage
del windows, fresh_nodes, labels_by_seed

## 3. Graph ablations

Refit the fixed architecture with patch-pooled loss on no-edge and
shuffled-destination graphs. The original shuffle uses the zero-based
outer-fold index as its RNG seed and preserves each source-edge list;
all five model seeds use that fold's same shuffled graph.

In [ ]:
stage = time.perf_counter()
_, base_graphs = lh.make_graphs(nodes, meta, graph_type="delaunay")
for mode in ["none", "shuffled"]:
    folds = []
    for fold, mouse in enumerate(MICE):
        control_graphs = lh.graph_control(base_graphs, mode, seed=fold)
        result = lh.run_outer_folds(
            control_graphs, meta, held_mice=[mouse], seeds=SEEDS,
            config=SELECTED["fixed_gnn"], objective="patch_sum",
            output_dir=CONTROL_RESULTS / PARTITION_MODE / f"graph_{mode}"
        )
        folds.append(result)
    result = lh.merge_outer_results(folds)
    lh.save_outer_outputs(result, CONTROL_RESULTS / PARTITION_MODE / f"graph_{mode}")
    CONTROL_METRICS.append(lh.metric_table(result["ensemble"], f"Graph {mode} · patch-pooled"))
TIMINGS["graph_ablations_80_fits"] = time.perf_counter() - stage
display(pd.concat(CONTROL_METRICS, ignore_index=True))
del base_graphs, control_graphs

## 4. Mouse-isolated image-to-cluster controls

For every held-out mouse, refit normalization bounds, the background
scaler and background K-means on the seven training mice. L3 clusters
the 12 marker summaries. L3b trains a selected seed-0 CNN on training
signal patches only. Fit the K=6 scaler and centroids on training signal,
then assign held-out patches to those centroids.

Each representation receives the fixed ROI-mean classifier with five
seeds, and composition LR with its retained fold-specific C. These
controls use complete fold-specific representations; the main notebook
uses a shared cohort partition.

In [ ]:
for representation, selection in [("marker_summary", "L3"), ("cnn", "L3b")]:
    stage = time.perf_counter()
    folds, lr_folds = [], []
    for mouse in MICE:
        fold_nodes, record = lh.fit_fold_partition(
            images, meta, held_mouse=mouse, representation=representation,
            config=SELECTED["cnn"],
            output_dir=CONTROL_RESULTS / f"fold_isolated_{selection}",
            device=DEVICE
        )
        _, fold_graphs = lh.make_graphs(fold_nodes, meta, graph_type="delaunay")
        result = lh.run_outer_folds(
            fold_graphs, meta, held_mice=[mouse], seeds=SEEDS,
            config=SELECTED["fixed_gnn"], objective="roi_mean",
            output_dir=CONTROL_RESULTS / f"fold_isolated_{selection}" / "prediction"
        )
        folds.append(result)
        lr_folds.append(lh.composition_lr(
            fold_graphs, meta, SELECTED["fold_isolated_composition_lr"][selection],
            held_mice=[mouse]
        ))
        del fold_nodes, fold_graphs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    result = lh.merge_outer_results(folds)
    lr_result = pd.concat(lr_folds, ignore_index=True)
    folder = CONTROL_RESULTS / f"fold_isolated_{selection}"
    lh.save_outer_outputs(result, folder / "prediction")
    lr_result.to_csv(folder / "composition_lr_predictions.csv", index=False)
    CONTROL_METRICS.extend([
        lh.metric_table(result["ensemble"], f"{selection} · mouse-isolated ROI-mean"),
        lh.metric_table(lr_result, f"{selection} · mouse-isolated composition LR")
    ])
    TIMINGS[f"fold_isolated_{selection}"] = time.perf_counter() - stage
    display(pd.concat(CONTROL_METRICS[-2:], ignore_index=True))

## 5. Export all control results

In [ ]:
metrics = pd.concat(CONTROL_METRICS, ignore_index=True)
metrics.to_csv(CONTROL_RESULTS / f"{PARTITION_MODE}_metrics.csv", index=False)
TIMINGS["notebook_total"] = time.perf_counter() - started
timing_table = pd.DataFrame({"stage": list(TIMINGS), "seconds": list(TIMINGS.values())})
timing_table.to_csv(CONTROL_RESULTS / "timings.csv", index=False)
lh.write_json(CONTROL_RESULTS / "run.json", {
    "partition_mode": PARTITION_MODE, "config": SELECTED,
    "library_versions": lh.environment(), "timings_seconds": TIMINGS,
})
display(metrics)
display(timing_table)